# XLM-RoBERTa Fine-Tuning Pipeline

Adım 1 : Veri ve ağırlıkları yükle

Adım 2 : PyTorch Dataset sınıfı

Adım 3 : Modeli indir ve yapılandır

Adım 4 : Özel eğitmen (ImbalancedTrainer)

Adım 5 : Eğitim parametreleri

Adım 6 : Eğitimi başlat

Adım 7 : Test seti ile final sınav + rapor

Gerekli İmportlar

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import json
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

In [2]:
# GENEL AYARLAR
MODEL_ADI       = "cardiffnlp/twitter-xlm-roberta-base-sentiment"
TOKENIZED_FILE  = "tokenized.pt"      # nlp_pipeline.py çıktısı
TRAIN_CSV       = "train.csv"
VAL_CSV         = "val.csv"
TEST_CSV        = "test.csv"
WEIGHTS_FILE    = "class_weights.json"
TEXT_COL        = "Yorum Metni"
LABEL_COL       = "etiket_id"
OUTPUT_DIR      = "xlm_roberta_model"
RAPOR_DIR       = "egitim_rapor"
MAX_LENGTH      = 80

ETIKET_ISIMLERI = {
    0: "olumlu",
    1: "olumsuz",
    2: "nötr",
    3: "öneri/tavsiye",
    4: "şikayet",
}
NUM_LABELS = len(ETIKET_ISIMLERI)

In [3]:
# Donanım tespiti
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Cihaz: {DEVICE}")
print(f"{'GPU: ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU modu'}")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(RAPOR_DIR,  exist_ok=True)

Cihaz: cpu
CPU modu


In [4]:
# ═════════════════════════════════════════════════════
# ADIM 1 : CEPHANEYİ TOPLAMA (Veri + Ağırlık Yükleme)
# ═════════════════════════════════════════════════════

In [5]:
# ── 1a. Tokenize edilmiş TÜM veriyi yükle (.pt dosyası)
print("  Tokenized veri seti yükleniyor...")
token_data = torch.load(TOKENIZED_FILE, map_location="cpu")

# Çekmecelerden verileri çıkar
train_data = token_data["train"]
val_data   = token_data["val"]
test_data  = token_data["test"]

print(f"  Train boyutu        : {train_data['input_ids'].shape[0]}")
print(f"  Val boyutu          : {val_data['input_ids'].shape[0]}")
print(f"  Test boyutu         : {test_data['input_ids'].shape[0]}")

  Tokenized veri seti yükleniyor...
  Train boyutu        : 5769
  Val boyutu          : 721
  Test boyutu         : 722


In [6]:
# ── 1b. Test CSV'sini yükle (Sadece Adım 7'deki raporlama için lazım)
print("  Test CSV yükleniyor (Raporlama için)...")
df_test = pd.read_csv(TEST_CSV)

  Test CSV yükleniyor (Raporlama için)...


In [7]:
# ── 1c. Sınıf ağırlıklarını yükle (class_weights.json)
print("  Sınıf ağırlıkları yükleniyor...")
with open(WEIGHTS_FILE, "r", encoding="utf-8") as f:
    weights_dict = json.load(f)

class_weights = torch.tensor(
    [weights_dict[str(i)] for i in range(NUM_LABELS)],
    dtype=torch.float,
).to(DEVICE)
print(f"  Sınıf ağırlıkları   : {class_weights.tolist()}")

  Sınıf ağırlıkları yükleniyor...
  Sınıf ağırlıkları   : [0.05504617840051651, 0.09331505745649338, 0.128743976354599, 0.48905715346336365, 4.233837604522705]


In [8]:
# ═════════════════════════════════════════════════════
# ADIM 2 : VERİYİ MODELE UYGUN FORMATA SOKMA
# ═════════════════════════════════════════════════════

In [9]:
class YorumDataset(Dataset):
    def __init__(self, veri_sozlugu):
        self.input_ids      = veri_sozlugu["input_ids"]
        self.attention_mask = veri_sozlugu["attention_mask"]
        self.labels         = veri_sozlugu["labels"]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids"     : self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels"        : self.labels[idx],
        }

# Artık CSV'den on-the-fly (anlık) tokenization yapmaya gerek yok!
# Doğrudan hazırladığımız tensörleri kutulara koyuyoruz:
train_dataset = YorumDataset(train_data)
val_dataset   = YorumDataset(val_data)
test_dataset  = YorumDataset(test_data)

print(f"  Train Dataset hazır.")
print(f"  Val   Dataset hazır.")

  Train Dataset hazır.
  Val   Dataset hazır.


In [10]:
# ═════════════════════════════════════════════════════
# ADIM 3 : BEYNİ İNDİRME (Model Kurulumu)
# ═════════════════════════════════════════════════════

In [11]:
print(f"  Model indiriliyor: {MODEL_ADI}")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ADI,
    num_labels=NUM_LABELS,
    ignore_mismatched_sizes=True,   # orijinal 3-sınıflı head'i yeni 5-sınıflıyla değiştir
)
model = model.to(DEVICE)


  Model indiriliyor: cardiffnlp/twitter-xlm-roberta-base-sentiment


[transformers] You passed `num_labels=5` which is incompatible to the `id2label` map of length `3`.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-xlm-roberta-base-sentiment
Key                        | Status   |                                                                                       
---------------------------+----------+---------------------------------------------------------------------------------------
classifier.out_proj.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3]) vs model:torch.Size([5])          
classifier.out_proj.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3, 768]) vs model:torch.Size([5, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [12]:
# Parametre sayısını hesapla
toplam_param    = sum(p.numel() for p in model.parameters())
egitilmis_param = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Toplam parametre    : {toplam_param:,}")
print(f"  Eğitilecek parametre: {egitilmis_param:,}")

  Toplam parametre    : 278,047,493
  Eğitilecek parametre: 278,047,493


In [13]:
# ═════════════════════════════════════════════════════
# ADIM 4 : ÖZEL EĞİTMEN SINIFI (ImbalancedTrainer)
# ═════════════════════════════════════════════════════

In [14]:
class ImbalancedTrainer(Trainer):
    """
    Hugging Face'in standart Trainer'ının üzerine yazılmış özel versiyon.
    Tek fark: loss hesaplanırken sınıf ağırlıkları devreye girer.
    Az görülen sınıflar (şikayet, öneri) yanlış bilgininde daha büyük
    ceza alır → model her şeye 'olumlu' diyemez.
    """

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # Etiketleri inputs'dan çıkar
        labels = inputs.pop("labels")

        # Modeli çalıştır
        outputs = model(**inputs)
        logits  = outputs.logits

        # Ağırlıklı Cross Entropy Loss
        loss_fn = nn.CrossEntropyLoss(weight=class_weights)
        loss    = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [15]:
# ═════════════════════════════════════════════════════
# ADIM 5 : EĞİTİM KURALLARINI BELİRLEME
# ═════════════════════════════════════════════════════

In [16]:
# Batch size — GPU varsa 32, yoksa 16 (bellek taşmasın)
BATCH_SIZE = 32 if torch.cuda.is_available() else 16
EPOCHS     = 4
LR         = 2e-5   # XLM-RoBERTa için standart öğrenme hızı

In [17]:
training_args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE * 2,   # val'de grad yok, 2x daha hızlı
    learning_rate               = LR,
    weight_decay                = 0.01,             # L2 regularizasyon (overfitting önler)
    lr_scheduler_type           = "cosine",         # epoch sonuna doğru LR düşer
    eval_strategy               = "epoch",          # her epoch sonunda val seti değerlendir
    save_strategy               = "epoch",          # her epoch sonunda checkpoint kaydet
    load_best_model_at_end      = True,             # en iyi val skoru olan modeli tut
    metric_for_best_model       = "eval_accuracy",
    warmup_steps                = 100,              # Ortalama bir adım sayısı
    greater_is_better           = True,
    logging_steps               = 50,               # her 50 batch'te bir log yaz
    save_total_limit            = 2,                # en fazla 2 checkpoint tut (disk tasarrufu)
    report_to                   = "none",           # wandb/tensorboard kapalı
    fp16                        = torch.cuda.is_available(),  # GPU varsa yarı hassasiyet
    dataloader_num_workers      = 0,                # Windows uyumluluğu için 0
    seed                        = 42,
)

In [18]:
print(f"  Epoch sayısı        : {EPOCHS}")
print(f"  Batch size          : {BATCH_SIZE}")
print(f"  Learning rate       : {LR}")
print(f"  Warmup oranı        : %10")
print(f"  Scheduler           : cosine")
print(f"  FP16 (yarı hassas)  : {torch.cuda.is_available()}")

  Epoch sayısı        : 4
  Batch size          : 16
  Learning rate       : 2e-05
  Warmup oranı        : %10
  Scheduler           : cosine
  FP16 (yarı hassas)  : False


In [19]:
# ── Değerlendirme metriği
def compute_metrics(eval_pred):
    """Her epoch sonunda Trainer bu fonksiyonu çağırır."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"eval_accuracy": accuracy_score(labels, preds)}

In [20]:
# ═════════════════════════════════════════════════════
# ADIM 6 : EĞİTİMİ BAŞLAT
# ═════════════════════════════════════════════════════

In [21]:
# ═════════════════════════════════════════════════════
# ADIM 6.1 : TRAINER NESNESİNİ OLUŞTURMA
# ═════════════════════════════════════════════════════
print("\n" + "═"*55)
print("ADIM 6.1 : Eğitim hazırlıkları (Trainer)")
print("═"*55)

trainer = ImbalancedTrainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    compute_metrics = compute_metrics,
    callbacks       = [
        EarlyStoppingCallback(
            early_stopping_patience=2,          # 2 epoch iyileşmezse dur
            early_stopping_threshold=0.001,
        )
    ],
)
print("  Trainer başarıyla oluşturuldu. Eğitime geçilebilir.")


═══════════════════════════════════════════════════════
ADIM 6.1 : Eğitim hazırlıkları (Trainer)
═══════════════════════════════════════════════════════
  Trainer başarıyla oluşturuldu. Eğitime geçilebilir.


In [22]:
# ═════════════════════════════════════════════════════
# ADIM 6.2 : EĞİTİMİ BAŞLAT (Bu işlem uzun sürebilir!)
# ═════════════════════════════════════════════════════
print("  trainer.train() çağrılıyor... Lütfen bitene kadar bekleyin.\n")

train_result = trainer.train()

# Eğitim istatistikleri
print(f"\n  Toplam eğitim süresi : {train_result.metrics['train_runtime']:.1f} sn")
print(f"  Örnek/saniye         : {train_result.metrics['train_samples_per_second']:.1f}")
print(f"  Son train loss       : {train_result.metrics['train_loss']:.4f}")

  trainer.train() çağrılıyor... Lütfen bitene kadar bekleyin.



C:\Users\05hde\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.566535,0.636565,0.833564
2,0.592012,0.619105,0.869626
3,0.227501,0.696599,0.883495
4,0.181688,0.721849,0.879334


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\05hde\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


...

C:\Users\05hde\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\05hde\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


  Toplam eğitim süresi : 5859.9 sn
  Örnek/saniye         : 3.9
  Son train loss       : 0.4220


In [24]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ADI)

In [25]:
# ═════════════════════════════════════════════════════
# ADIM 6.3 : EN İYİ MODELİ KAYDET
# ═════════════════════════════════════════════════════
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\n  Model klasöre kaydedildi: {OUTPUT_DIR}/")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


  Model klasöre kaydedildi: xlm_roberta_model/


In [26]:
# ═════════════════════════════════════════════════════
# ADIM 6.4 : EĞİTİM GRAFİKLERİNİ ÇİZ
# ═════════════════════════════════════════════════════
log_history = trainer.state.log_history
train_losses = [x["loss"]       for x in log_history if "loss"      in x]
val_losses   = [x["eval_loss"]  for x in log_history if "eval_loss" in x]
val_accs     = [x["eval_accuracy"] for x in log_history if "eval_accuracy" in x]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(train_losses, label="Train Loss", color="#3498DB")

if val_losses:
    xs = [int(len(train_losses) / len(val_losses)) * (i+1) for i in range(len(val_losses))]
    axes[0].plot(xs, val_losses, label="Val Loss", color="#E74C3C", marker="o")
axes[0].set_title("Loss Eğrisi")
axes[0].set_xlabel("Adım")
axes[0].set_ylabel("Loss")
axes[0].legend()

if val_accs:
    axes[1].plot(val_accs, marker="o", color="#2ECC71")
    axes[1].set_title("Validation Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].set_ylim(0, 1)
    for i, v in enumerate(val_accs):
        axes[1].annotate(f"%{v*100:.1f}", (i, v), textcoords="offset points", xytext=(0, 8), ha="center")

fig.tight_layout()
fig.savefig(os.path.join(RAPOR_DIR, "egitim_egrisi.png"), dpi=150)
print(f"  Grafik kaydedildi: {RAPOR_DIR}/egitim_egrisi.png")

# Jupyter'de grafiği hücre altında görmek için:
fig

  Grafik kaydedildi: egitim_rapor/egitim_egrisi.png


<Figure size 1200x400 with 2 Axes>

In [27]:
# ═════════════════════════════════════════════════════
# ADIM 7 : FİNAL SINAVI VE RAPORLAMA (Test Seti)
# ═════════════════════════════════════════════════════

In [28]:
# ═════════════════════════════════════════════════════
# ADIM 7.1 : TEST SETİ TAHMİNLERİ VE RAPORLAMA
# ═════════════════════════════════════════════════════
print("\n" + "═"*55)
print("ADIM 7.1 : Final sınav (test seti)")
print("═"*55)

# En iyi modeli yükle (load_best_model_at_end=True zaten yükledi)
print("  Test seti tahmin ediliyor...")
test_pred    = trainer.predict(test_dataset)
test_logits  = test_pred.predictions
test_labels  = test_pred.label_ids
test_preds   = np.argmax(test_logits, axis=-1)

# ── Genel doğruluk
test_acc = accuracy_score(test_labels, test_preds)
print(f"\n  Test Accuracy: %{test_acc*100:.2f}")

# ── Sınıflandırma raporu
isim_listesi = [ETIKET_ISIMLERI[i] for i in range(NUM_LABELS)]
rapor = classification_report(
    test_labels, test_preds,
    target_names=isim_listesi,
    digits=3,
)
print("\n  SINIFLANDIRMA RAPORU:")
print(rapor)

# Raporu dosyaya kaydet
with open(os.path.join(RAPOR_DIR, "classification_report.txt"), "w", encoding="utf-8") as f:
    f.write(f"Test Accuracy: %{test_acc*100:.2f}\n\n")
    f.write(rapor)
print(f"  Rapor kaydedildi: {RAPOR_DIR}/classification_report.txt")


═══════════════════════════════════════════════════════
ADIM 7.1 : Final sınav (test seti)
═══════════════════════════════════════════════════════
  Test seti tahmin ediliyor...


C:\Users\05hde\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



  Test Accuracy: %87.53

  SINIFLANDIRMA RAPORU:
               precision    recall  f1-score   support

       olumlu      0.926     0.932     0.929       337
      olumsuz      0.845     0.824     0.835       199
         nötr      0.806     0.806     0.806       144
öneri/tavsiye      0.923     0.947     0.935        38
      şikayet      0.333     0.500     0.400         4

     accuracy                          0.875       722
    macro avg      0.767     0.802     0.781       722
 weighted avg      0.876     0.875     0.876       722

  Rapor kaydedildi: egitim_rapor/classification_report.txt


In [29]:
# ═════════════════════════════════════════════════════
# ADIM 7.2 : KARMAŞIKLIK MATRİSİ (CONFUSION MATRIX)
# ═════════════════════════════════════════════════════
cm = confusion_matrix(test_labels, test_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)   # satır bazında normalize

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Ham sayı matrisi
sns.heatmap(
    cm, annot=True, fmt="d",
    xticklabels=isim_listesi, yticklabels=isim_listesi,
    cmap="Blues", ax=axes[0],
)
axes[0].set_title("Karmaşıklık Matrisi (Ham Sayı)")
axes[0].set_xlabel("Tahmin")
axes[0].set_ylabel("Gerçek")
axes[0].tick_params(axis="x", rotation=30)

# Normalize edilmiş matris (%)
sns.heatmap(
    cm_norm, annot=True, fmt=".2%",
    xticklabels=isim_listesi, yticklabels=isim_listesi,
    cmap="Greens", ax=axes[1],
    vmin=0, vmax=1,
)
axes[1].set_title("Karmaşıklık Matrisi (Normalize %)")
axes[1].set_xlabel("Tahmin")
axes[1].set_ylabel("Gerçek")
axes[1].tick_params(axis="x", rotation=30)

fig.tight_layout()
fig.savefig(os.path.join(RAPOR_DIR, "confusion_matrix.png"), dpi=150)
print(f"  Grafik kaydedildi: {RAPOR_DIR}/confusion_matrix.png")

# Jupyter'de grafiği görüntülemek için:
fig

  Grafik kaydedildi: egitim_rapor/confusion_matrix.png


<Figure size 1400x600 with 4 Axes>

In [30]:
# ═════════════════════════════════════════════════════
# ADIM 7.3 : F1 SKORLARI GRAFİĞİ
# ═════════════════════════════════════════════════════
rapor_dict = classification_report(
    test_labels, test_preds,
    target_names=isim_listesi,
    output_dict=True,
)
f1_scores = [rapor_dict[k]["f1-score"] for k in isim_listesi]
renkler   = ["#2ECC71", "#E74C3C", "#95A5A6", "#3498DB", "#E67E22"]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(isim_listesi, f1_scores, color=renkler, edgecolor="white")
ax.set_xlim(0, 1)
ax.set_xlabel("F1 Score")
ax.set_title("Sınıf Bazında F1 Skorları")
for bar, val in zip(bars, f1_scores):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
            f"{val:.3f}", va="center", fontsize=10)
            
fig.tight_layout()
fig.savefig(os.path.join(RAPOR_DIR, "f1_scores.png"), dpi=150)
print(f"  Grafik kaydedildi: {RAPOR_DIR}/f1_scores.png")

# Jupyter'de grafiği görüntülemek için:
fig

  Grafik kaydedildi: egitim_rapor/f1_scores.png


<Figure size 800x400 with 1 Axes>

In [31]:
# ═════════════════════════════════════════════════════
# ADIM 7.4 : CSV DIŞA AKTARIM VE EĞİTİM ÖZETİ
# ═════════════════════════════════════════════════════
# ── Tahminleri CSV'ye ekle
df_test["tahmin_id"]     = test_preds
df_test["tahmin_etiket"] = [ETIKET_ISIMLERI[p] for p in test_preds]
df_test["dogru_mu"]      = (df_test[LABEL_COL] == df_test["tahmin_id"]).map({True: "evet", False: "hayır"})
df_test.to_csv(os.path.join(RAPOR_DIR, "test_tahminler.csv"), index=False, encoding="utf-8-sig")
print(f"  Test tahminleri kaydedildi: {RAPOR_DIR}/test_tahminler.csv")

  Test tahminleri kaydedildi: egitim_rapor/test_tahminler.csv


In [32]:
# ═════════════════════════════════════════════════════
# ÖZET
# ═════════════════════════════════════════════════════
print("\n" + "═"*55)
print("EĞİTİM TAMAMLANDI")
print("═"*55)
print(f"  Test Accuracy        : %{test_acc*100:.2f}")
print(f"  Model dizini         : {OUTPUT_DIR}/")
print(f"  Raporlar             : {RAPOR_DIR}/")
print(f"    - egitim_egrisi.png")
print(f"    - confusion_matrix.png")
print(f"    - f1_scores.png")
print(f"    - classification_report.txt")
print(f"    - test_tahminler.csv")
print("═"*55)
print("\nModeli kullanmak için:")
print('  from transformers import pipeline')
print(f'  clf = pipeline("text-classification", model="{OUTPUT_DIR}")')
print('  clf("Bu video harikaydı!")')


═══════════════════════════════════════════════════════
EĞİTİM TAMAMLANDI
═══════════════════════════════════════════════════════
  Test Accuracy        : %87.53
  Model dizini         : xlm_roberta_model/
  Raporlar             : egitim_rapor/
    - egitim_egrisi.png
    - confusion_matrix.png
    - f1_scores.png
    - classification_report.txt
    - test_tahminler.csv
═══════════════════════════════════════════════════════

Modeli kullanmak için:
  from transformers import pipeline
  clf = pipeline("text-classification", model="xlm_roberta_model")
  clf("Bu video harikaydı!")
